# ⚖️ DỰ ÁN VILAW-LLM: HUẤN LUYỆN DPO ALIGNMENT (DIRECT PREFERENCE OPTIMIZATION)
### Khử Ảo Giác (Hallucination) & Căn Chỉnh Lập Luận Pháp Lý Chuyên Sâu

> **Quy trình thực hiện trên Colab:**
> 1. Tải lên 2 file: `vilaw-sft-lora.zip` và `legal_dpo_pairs.parquet`.
> 2. Chạy lần lượt các Cell bên dưới để huấn luyện và kiểm thử mô hình DPO.

## 1. Kiểm tra GPU, Cài đặt Unsloth & Giải nén SFT Adapter

In [ ]:
# 1.1. Kiểm tra GPU
!nvidia-smi

In [ ]:
# 1.2. Cài đặt thư viện Unsloth và TRL tối ưu riêng cho Colab T4
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes
!pip install pyarrow pandas datasets

In [ ]:
# 1.3. Giải nén an toàn: Tạo thẳng thư mục vilaw-sft-lora và bung file vào đó
import os

# Kiểm tra các file zip có trong thư mục hiện tại
zip_files = [f for f in os.listdir('.') if f.endswith('.zip') and 'sft' in f.lower()]
if zip_files:
    target_zip = zip_files[0]
    print(f"-> Đang giải nén: {target_zip} vào thư mục vilaw-sft-lora...")
    !mkdir -p vilaw-sft-lora
    !unzip -o -q "$target_zip" -d vilaw-sft-lora
    print("✓ Giải nén hoàn tất!")
else:
    print("ℹ️ Không tìm thấy file zip SFT, kiểm tra các thư mục hiện có...")

# In ra danh sách file để kiểm tra
!ls -la

## 2. Nạp SFT Model từ Sprint 3 (Tự động dò tìm đường dẫn Adapter)

In [ ]:
import os
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch

# Kích hoạt tăng tốc DPO cho Unsloth (tiết kiệm 50% VRAM)
PatchDPOTrainer()

# Thuật toán tự động tìm thư mục chứa adapter_config.json
adapter_dir = None
for root, dirs, files in os.walk("."):
    if "adapter_config.json" in files:
        adapter_dir = root
        break

if adapter_dir is None:
    raise FileNotFoundError(
        "❌ Vẫn chưa tìm thấy file adapter_config.json! "
        "Hãy chắc chắn bạn đã upload file vilaw-sft-lora.zip lên Colab và chạy ô Cell 1.3 ở trên."
    )

print(f"🎯 ĐÃ TÌM THẤY SFT LORA ADAPTER TẠI: '{adapter_dir}'")

max_seq_length = 2048
print(f"Đang nạp mô hình từ: {adapter_dir}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_dir,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
print("✓ Nạp mô hình thành công!")

## 3. Chuẩn bị Dữ liệu DPO (Prompt, Chosen, Rejected)

In [ ]:
import pandas as pd
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

SYSTEM_PROMPT = (
    "Bạn là một chuyên gia tư vấn pháp luật Việt Nam am hiểu sâu sắc các quy định pháp luật. "
    "Hãy trả lời câu hỏi dựa trên các văn bản quy phạm pháp luật hiện hành, "
    "viện dẫn chính xác số Điều, Khoản, tên luật và đưa ra lập luận logic, rõ ràng."
)

# Đọc file legal_dpo_pairs.parquet
df_dpo = pd.read_parquet("legal_dpo_pairs.parquet")
print(f"✓ Đã nạp {len(df_dpo):,} cặp dữ liệu DPO!")

formatted_dpo = []
for _, row in df_dpo.iterrows():
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"]}
    ]
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    formatted_dpo.append({
        "prompt": prompt_text,
        "chosen": row["chosen"],
        "rejected": row["rejected"]
    })

dpo_dataset = Dataset.from_pandas(pd.DataFrame(formatted_dpo))
print("\nVí dụ 1 mẫu DPO chuẩn bị train:")
print("PROMPT:", dpo_dataset[0]["prompt"][:200])
print("\nCHOSEN:", dpo_dataset[0]["chosen"][:200])
print("\nREJECTED:", dpo_dataset[0]["rejected"][:200])

## 4. Cấu hình DPOConfig & Bắt đầu Huấn luyện DPOTrainer

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir="./vilaw-dpo-checkpoints",
    beta=0.1,                       # Hệ số KL penalty
    learning_rate=5e-6,             # LR nhỏ để căn chỉnh nhẹ nhàng
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16, # Effective batch size = 16
    num_train_epochs=1,
    max_prompt_length=512,
    max_length=1536,
    optim="adamw_8bit",
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=42,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,                 # Tự động đóng băng base model làm reference!
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    args=dpo_config,
)

print("🚀 Bắt đầu huấn luyện DPO Alignment...")
dpo_trainer.train()

## 5. Kiểm thử Suy luận & Đóng gói DPO Adapter để Tải về

In [ ]:
# 5.1. Chuyển sang chế độ Inference
FastLanguageModel.for_inference(model)

cau_hoi_hoc_bua = "Công ty tư nhân có bắt buộc phải lập Ban Kiểm soát hay không và căn cứ vào quy định nào của Luật Doanh nghiệp?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": cau_hoi_hoc_bua}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.2,
    repetition_penalty=1.15
)

tra_loi_dpo = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== CÂU TRẢ LỜI CỦA VILAW-LLM SAU KHI QUA DPO ALIGNMENT ===\n")
print(tra_loi_dpo)

In [ ]:
# 5.2. Lưu DPO Adapter & Nén file zip để tải về máy
model.save_pretrained("vilaw-dpo-lora")
tokenizer.save_pretrained("vilaw-dpo-lora")
!zip -r vilaw-dpo-lora.zip vilaw-dpo-lora
print("\n✅ HOÀN TẤT! File 'vilaw-dpo-lora.zip' đã sẵn sàng trong thư mục Colab.")
print("👉 Hãy nhấp chuột phải vào file vilaw-dpo-lora.zip và bấm Download để tải về máy!")

In [ ]:
import shutil
shutil.make_archive("vilaw-dpo-lora", 'zip', "vilaw-dpo-lora")
from google.colab import files
files.download("vilaw-dpo-lora.zip")